# Yaw Rate Steering Controller — 내부 신호 검증

**목적**: 요레이트 조향 제어기의 명령 대 실제 차량 응답을 폐루프 시뮬레이션으로 확인한다.

**시나리오**: 사인 파형 요레이트 지령 — 초기 속도 30 kph, 이후 구동 입력 없음.

**확인 항목**
| 항목 | 설명 |
|---|---|
| Yaw Rate | 요레이트 추종 [rad/s] |
| Mz | 요 모멘트 실제 [N·m] |
| Fy_total | 총 횡력 실제 [N] |
| delta | 조향각 명령 vs 실제 [rad] |
| Torque | 조향 토크 명령 [N·m] |

## 1. 환경 설정

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT_DIR = Path().resolve().parents[3]
sys.path.insert(0, str(ROOT_DIR))

VEHICLE_CONFIG_PATH   = ROOT_DIR / "vehicle_sim" / "models" / "params" / "vehicle_standard.yaml"
CONTROLLER_PARAM_PATH = ROOT_DIR / "vehicle_sim" / "controllers" / "yaw_rate_steering_controller" / "param" / "controller_options.example.yaml"

## 2. 모듈 로드

- **`YawRateSineScenario`**: 사인 파형 요레이트 지령 시나리오 (`vehicle_sim/scenarios/yaw_rate_sine`).
- **`YawRateSteeringController`**: 요레이트 기반 조향 제어기.
- **`yaw_sim_utils`**: 차량 상태 변환·측정 헬퍼 함수.

In [ ]:
from vehicle_sim.scenarios.yaw_rate_sine import generate
from vehicle_sim.controllers.yaw_rate_steering_controller import load_controller_runtime_config
from vehicle_sim.utils.yaw_sim_utils import (
    build_yaw_sim, init_sim_log, log_step,
    build_controller_state, build_corner_inputs, measure_vehicle_outputs,
)

runtime_cfg   = load_controller_runtime_config(CONTROLLER_PARAM_PATH)
controller_dt = float(runtime_cfg.options.dt)

print('runtime mode :', runtime_cfg.mode)
print('controller dt:', controller_dt)

## 3. 시뮬레이션

**루프 구조**

| 단계 | 내용 |
|---|---|
| 센서 읽기 (`state`) | 차량 상태 → 제어기 입력 형식으로 변환 |
| 지령 (`reference`) | 시나리오에서 요레이트 목표 조회 |
| 제어기 업데이트 | `steer.update(state, reference)` / `trq.update(state, reference)` |
| 모델 업데이트 | 차량 물리 모델 진행 |

In [ ]:
scenario = generate(dt=controller_dt)


def run(scenario, runtime_cfg) -> dict:
    """폐루프 시뮬레이션 — 요레이트 추종."""
    dt = controller_dt

    steer, trq, body, wheels = build_yaw_sim(runtime_cfg, str(VEHICLE_CONFIG_PATH), scenario)

    t   = scenario['time']
    n   = len(t)
    log = init_sim_log(n, t, wheels)

    for i in range(n):

        # 센서 읽기 — 차량 상태
        state = build_controller_state(body)

        # 지령 (reference)
        reference = {'yaw_rate': scenario.yaw_rate_ref(i)}

        # steer.update — 조향각 지령
        delta_cmd = steer.update(state, reference)

        # trq.update — 조향 토크 지령
        steer_trq = trq.update(state, reference)

        # 물리 모델 진행
        body.update(dt, build_corner_inputs(body, steer_trq, 0.0), direction=1)
        out = measure_vehicle_outputs(body)

        # 기록
        log_step(log, i, body, reference['yaw_rate'], out, delta_cmd, steer_trq, wheels)

    return log


log = run(scenario, runtime_cfg)
print(f"Done: {len(log['time'])} steps")

## 4. 결과 시각화

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(12, 16), sharex=True)

wheels = list(log['delta_meas'].keys())

axes[0].plot(log['time'], log['yaw_ref'],  '--', label='Reference [rad/s]', alpha=0.8)
axes[0].plot(log['time'], log['yaw_meas'],       label='Measured [rad/s]')
axes[0].set_title('Yaw Rate Tracking')
axes[0].set_ylabel('rad/s')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(log['time'], log['yaw_ref'] - log['yaw_meas'], label='Error [rad/s]')
axes[1].axhline(0.0, color='k', linewidth=0.8, alpha=0.3)
axes[1].set_title('Yaw Rate Error')
axes[1].set_ylabel('rad/s')
axes[1].grid(alpha=0.3)
axes[1].legend()

axes[2].plot(log['time'], log['mz_actual'], label='Mz_actual [N·m]', linewidth=2.0)
axes[2].set_title('Yaw Moment (Actual)')
axes[2].set_ylabel('N·m')
axes[2].grid(alpha=0.3)
axes[2].legend()

axes[3].plot(log['time'], log['fy_total_actual'], label='Fy_actual [N]', linewidth=2.0)
axes[3].set_title('Total Lateral Force (Actual)')
axes[3].set_ylabel('N')
axes[3].grid(alpha=0.3)
axes[3].legend()

for w in wheels:
    axes[4].plot(log['time'], log['delta_cmd'][w],  '--', label=f'{w} cmd [rad]', alpha=0.8)
    axes[4].plot(log['time'], log['delta_meas'][w],       label=f'{w} meas [rad]')
axes[4].set_title('Steering Angle: Command vs Measured')
axes[4].set_ylabel('rad')
axes[4].grid(alpha=0.3)
axes[4].legend(ncol=2)

for w in wheels:
    axes[5].plot(log['time'], log['steer_trq'][w], label=f'{w} [N·m]')
axes[5].set_title('Steering Torque Command')
axes[5].set_xlabel('Time [s]')
axes[5].set_ylabel('N·m')
axes[5].grid(alpha=0.3)
axes[5].legend(ncol=2)

plt.tight_layout()
plt.show()